# LFM2.5 Model Quantization (F16 → Q4_K_M)

**Purpose**: Quantize your fixed F16 GGUF to Q4_K_M (~220MB) for production use.

**Steps**:
1. Run all cells in order
2. Download the output file `lfm25_fixed_Q4_K_M.gguf`
3. Upload to HuggingFace

**Time**: ~5 minutes

In [ ]:
# Step 1: Clone and build llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && make -j$(nproc)

In [ ]:
# Step 2: Upload your F16 GGUF file
# Click the folder icon on the left → upload lfm25_fixed_f16.gguf
# Or run this to download from your local machine if you've shared it

from google.colab import files
print("Upload your lfm25_fixed_f16.gguf file (679MB)...")
uploaded = files.upload()
print("✓ Upload complete!")

In [ ]:
# Step 3: Verify file exists
!ls -lh lfm25_fixed_f16.gguf

In [ ]:
# Step 4: Quantize F16 → Q4_K_M
!./llama.cpp/llama-quantize lfm25_fixed_f16.gguf lfm25_fixed_Q4_K_M.gguf Q4_K_M

In [ ]:
# Step 5: Verify output
!ls -lh lfm25_fixed_Q4_K_M.gguf
print("\n✓ Quantization complete!")
print("Expected size: ~220MB")
print("Download the file below ↓")

In [ ]:
# Step 6: Download quantized model
from google.colab import files
files.download('lfm25_fixed_Q4_K_M.gguf')

---

## Next Steps After Download

### 1. Test on Device (Optional but Recommended)
```bash
# Push to device
adb push lfm25_fixed_Q4_K_M.gguf /sdcard/test.gguf
adb shell run-as com.hiva.runtime cp /sdcard/test.gguf files/models/lfm25/model.gguf

# Restart and check logs
adb shell am force-stop com.hiva.runtime
adb shell am start -n com.hiva.runtime/.MainActivity
adb logcat -s EdgeBrain:* LiquidInferenceEngine:*
```

**Expected**: `LEAP model loaded in XXXXms` (no crash)

### 2. Upload to HuggingFace
```bash
# Install HF CLI
pip install huggingface-hub

# Login
huggingface-cli login

# Upload (replace existing broken file)
huggingface-cli upload \
  Kenzlejaze/hiva-medichat-v2-gguf \
  lfm25_fixed_Q4_K_M.gguf \
  lfm25_350m_medichat_v2_merged.Q4_K_M.gguf \
  --repo-type model
```

### 3. Update App Code
Edit `src/services/modelDownloader.ts`:
```typescript
// REVERT from TinyLlama back to LFM2.5
const LEAP_MODEL_CONFIG = {
  url: 'https://huggingface.co/Kenzlejaze/hiva-medichat-v2-gguf/resolve/main/lfm25_350m_medichat_v2_merged.Q4_K_M.gguf',
  filename: 'model.gguf',
  expectedSizeMB: 219,
  expectedSizeBytes: 229_311_776,
  path: 'models/lfm25',
};
```

### 4. Rebuild and Deploy
```bash
npx vite build
npx cap sync android
cd android && ./gradlew assembleDebug
adb install -r app/build/outputs/apk/debug/app-debug.apk

# Remove old model, let app download fixed version
adb shell run-as com.hiva.runtime rm files/models/lfm25/model.gguf
```